# Débogage DINO — zone de 512 × 512 px, mode `tangent`

Tout se joue sur une zone de **`ZONE_PX × ZONE_PX` pixels** (512 par défaut, soit une cellule du niveau `LEVEL - 9`).
DINO y travaille sur de **petites images** : une par cellule de `PARENT_LEVEL`, de `TILE_PX` pixels de côté (64 par défaut).
Par défaut : cellules de 32 px (`PARENT_LEVEL = LEVEL - 5`) dans des images de 64 px, soit 4 × 4 tokens par image et 16 × 16 images dans la zone.
Une cellule HEALPix n'est pas un carré nord-sud : pour qu'elle tienne entière dans son image, il faut une image plus grande
que la cellule. C'est ce que mesure le test 1b.

| Test | Ce qu'on regarde |
|---|---|
| **1** | La zone en haut, et en dessous les images envoyées à DINO dans la même disposition, avec le contour de leur cellule. |
| **1b** | Contrôle chiffré : chaque pixel d'image comparé à la cellule HEALPix sous lui, et la part de chaque cellule qui tombe dans son image. |
| **1c** | Zoom sur une image : grille des patchs 16 px, nord en haut. |
| **2** | `over_sample` : la fenêtre décalée de `(dy, dx)` px doit être l'image de base déplacée vers le **sud / l'est**. |
| **3** | `over_sample` de bout en bout avec `GetDINOV3SAT` : cartes de tokens pour `n = 1, 2, 4`, avec le code du package et, s'il est fautif, avec le décalage corrigé. D'abord avec un backbone exact (token = couleur moyenne du patch), puis avec DINOv3 si `WEIGHTS` est renseigné. |

Le notebook ne modifie pas le package. La version « corrigée » du test 3 remplace `tangent_grid_lonlat` le temps du calcul,
et seulement si le test 2 a trouvé le décalage inversé.


## Paramètres

In [ ]:
SOURCE        = "g4e"        # "g4e" : produits GRID4EARTH  |  "synthetic" : motif connu, sans réseau
PRODUCT_INDEX = 0            # quel produit de G4E_PRODUCTS
LEVEL         = 20           # niveau HEALPix des données (~6 m)
ZONE_PX       = 512          # côté de la zone, en pixels de LEVEL (puissance de 2)
PARENT_LEVEL  = 15           # une image DINO par cellule de ce niveau, cellule de 2**(LEVEL-PARENT_LEVEL) px
                             #   15 -> cellules de 32 px, 16 x 16 images dans la zone
                             #   14 -> cellules de 64 px, 8 x 8 images
TILE_PX       = 64           # côté des images envoyées à DINO (multiple de 16)
                             #   64 avec des cellules de 32 px : la cellule tient entière dans son image
                             #   64 avec des cellules de 64 px : ~25 % de chaque cellule déborde (voir test 1b)
                             #   None = règle par défaut de GetDINOV3SAT (pow2_cover_px)
OVER_SAMPLES  = (1, 2, 4)    # facteurs testés au test 3
WEIGHTS       = None         # chemin des poids DINOv3 SAT (.pth) ou id HF ; None = test 3 sans DINOv3
MODEL_NAME    = "dinov3_vitl16"
ZONE_LONLAT   = None         # (lon, lat) en degrés pour imposer la zone ; None = zone la plus complète
CACHE         = "cache_g4e"  # réflectances sur disque (None pour désactiver)
STRETCH       = (2, 98)      # percentiles d'affichage, communs à TOUTES les vignettes
ZOOM_TILE     = None         # (row, col) de l'image à zoomer ; None = centre de la grille


In [ ]:
import sys, os, warnings
from contextlib import contextmanager
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

sys.path.insert(0, os.path.abspath(".."))      # healpix_analyse depuis le dépôt
sys.path.insert(0, os.path.abspath("."))       # g4e_source à côté du notebook
import healpix_analyse.dino as D

ZONE_LEVEL = LEVEL - int(np.log2(ZONE_PX))
k = LEVEL - PARENT_LEVEL                       # cellule parente = 2**k px
GRID = 2 ** (PARENT_LEVEL - ZONE_LEVEL)        # la zone fait GRID x GRID images
assert ZONE_PX & (ZONE_PX - 1) == 0 and ZONE_LEVEL <= PARENT_LEVEL
assert k >= D.DINOV3_PATCH_LEVELS, "LEVEL - PARENT_LEVEL >= 4 en mode tangent"
print(f"données niveau {LEVEL} (~{D.healpix_gsd_m(LEVEL):.1f} m) | zone niveau {ZONE_LEVEL} : "
      f"{ZONE_PX} px ~{D.healpix_gsd_m(ZONE_LEVEL)/1e3:.2f} km | cellules DINO niveau {PARENT_LEVEL} : "
      f"{2**k} px ~{D.healpix_gsd_m(PARENT_LEVEL):.0f} m, {GRID}x{GRID} dans la zone")

## Données

In [ ]:
def synthetic_source(level, zone_level, lon0=2.35, lat0=48.85):
    # Motif connu, défini dans un repère métrique EST / NORD, à l'échelle de la zone :
    #   routes N-S (blanches) et E-O (jaunes), disques rouges, une flèche bleue vers le NORD.
    # Sur une image correcte : routes N-S verticales, E-O horizontales, flèche vers le haut.
    zone = int(D.cells_at_lonlat(zone_level, np.array([lon0]), np.array([lat0]))[0])
    nb = D.cell_neighbours(zone_level, np.array([zone]))[:, 0]
    blocks = np.r_[zone, nb[nb >= 0]]
    kk = level - zone_level
    ids = ((blocks[:, None] << (2 * kk)) + np.arange(4 ** kk)[None]).ravel()
    lon, lat = D.cell_centres_lonlat(level, ids)
    zlon, zlat = D.cell_centres_lonlat(zone_level, np.array([zone]))
    xi, eta = D.lonlat_to_offsets(zlon, zlat, lon[None], lat[None])
    X, Y = xi[0] * D.EARTH_RADIUS_M, eta[0] * D.EARTH_RADIUS_M     # mètres, est / nord
    L = D.healpix_gsd_m(zone_level); g = D.healpix_gsd_m(level)
    rgb = np.empty((ids.size, 3), np.float32)
    rgb[:] = (0.08, 0.14, 0.06)
    rgb[:, 1] += 0.04 * np.sin(X / (0.09 * L)) * np.cos(Y / (0.12 * L))
    sx, sy = L / 6.5, L / 4.3
    rgb[np.abs((X + sx / 2) % sx - sx / 2) < 2.5 * g] = (0.35, 0.35, 0.35)
    rgb[np.abs((Y + sy / 2) % sy - sy / 2) < 1.8 * g] = (0.30, 0.28, 0.08)
    for cx, cy, r in [(0.20, 0.25, 0.04), (-0.30, 0.10, 0.06), (0.05, -0.30, 0.05), (-0.15, -0.20, 0.03)]:
        rgb[np.hypot(X - cx * L, Y - cy * L) < r * L] = (0.35, 0.06, 0.05)
    a = 0.12 * L
    shaft = (np.abs(X) < a * 0.25) & (Y > -a) & (Y < a)
    head = (Y >= a) & (Y < 2 * a) & (np.abs(X) < (2 * a - Y) * 0.8)
    rgb[shaft | head] = (0.05, 0.10, 0.40)
    rgb[np.hypot(X - 0.35 * L, Y + 0.35 * L) < 0.03 * L] = np.nan  # un trou : teste fill="mean"

    class S: pass
    s = S(); s.level, s.cell_id, s.ellipsoid, s.dates = level, ids, "sphere", ["synthétique"]
    s.rgb = lambda i, **kw: rgb
    s.zone_lonlat = (lon0, lat0)
    return s

if SOURCE == "g4e":
    from g4e_source import ProductSeries, G4E_PRODUCTS
    src = ProductSeries(G4E_PRODUCTS, level=LEVEL)
    rgb_all = src.rgb(PRODUCT_INDEX, cache=CACHE)
else:
    src = synthetic_source(LEVEL, ZONE_LEVEL)
    rgb_all = src.rgb(0)

D.set_ellipsoid(getattr(src, "ellipsoid", "sphere"))
ids_all = np.asarray(src.cell_id, dtype=np.int64)
print(f"{ids_all.size:,} cellules, ellipsoïde {D.HEALPIX_ELLIPSOID}")

## Choix de la zone et sous-ensemble

On garde la zone et ses 8 voisines du même niveau : les images DINO débordent de leur cellule.

In [ ]:
finite = np.isfinite(rgb_all).all(axis=1)
zk = 2 * (LEVEL - ZONE_LEVEL)
if ZONE_LONLAT is None and SOURCE == "synthetic":
    ZONE_LONLAT = src.zone_lonlat                          # la zone où est dessiné le motif
if ZONE_LONLAT is not None:
    zone = int(D.cells_at_lonlat(ZONE_LEVEL, np.array([ZONE_LONLAT[0]]), np.array([ZONE_LONLAT[1]]))[0])
else:
    zid, cnt = np.unique(ids_all[finite] >> zk, return_counts=True)
    full = cnt / 4 ** (LEVEL - ZONE_LEVEL)
    zl, za = D.cell_centres_lonlat(ZONE_LEVEL, zid)
    order = np.lexsort((np.hypot(zl - np.median(zl), za - np.median(za)), -np.round(full, 3)))
    zone = int(zid[order[0]])

znb = D.cell_neighbours(ZONE_LEVEL, np.array([zone]))[:, 0]
sel = np.isin(ids_all >> zk, np.r_[zone, znb[znb >= 0]])
ids, rgb = ids_all[sel], rgb_all[sel]
in_zone = (ids >> zk) == zone

g = PARENT_LEVEL - ZONE_LEVEL
kids = (zone << (2 * g)) + np.arange(GRID * GRID)                   # les cellules DINO de la zone
x, y = D.nested_to_xy(kids - (zone << (2 * g)))
ROW, COL = GRID - 1 - y, x                                          # convention de nested_to_tiles
zlon, zlat = D.cell_centres_lonlat(ZONE_LEVEL, np.array([zone]))
clon, clat = D.cell_centres_lonlat(PARENT_LEVEL, kids)
print(f"zone {zone} (niveau {ZONE_LEVEL}), centre {zlon[0]:.4f}E {zlat[0]:.4f}N, "
      f"couverture {np.isfinite(rgb[in_zone]).all(1).mean():.1%}")

lo, hi = np.nanpercentile(rgb[in_zone], STRETCH, axis=0)            # étirement commun
def show(a, chw=True):
    a = np.transpose(a, (1, 2, 0)) if chw else a
    return np.clip((np.nan_to_num(a, nan=float(lo.mean())) - lo) / (hi - lo), 0, 1)

## Les images DINO — mêmes primitives et mêmes paramètres que `GetDINOV3SAT(projection="tangent")`

In [ ]:
gsd = D.healpix_gsd_m(LEVEL)
px = gsd / D.EARTH_RADIUS_M
H = W = int(TILE_PX) if TILE_PX is not None else D.pow2_cover_px(kids, PARENT_LEVEL, gsd)
assert H % D.DINOV3_PATCH == 0

def tangent_tiles_for(centres, shift=(0.0, 0.0), fill="mean"):
    # D.tangent_tiles restreint aux centres demandés, par paquets de ~2e6 pixels
    centres = np.asarray(centres)
    cl, ca = D.cell_centres_lonlat(PARENT_LEVEL, centres)
    M, chunk = len(centres), max(1, 2_000_000 // (H * W))
    vals = np.empty((M, H, W, 3), np.float32); valid = np.empty((M, H, W), bool)
    lon = np.empty((M, H, W)); lat = np.empty((M, H, W))
    for a in range(0, M, chunk):
        b = min(a + chunk, M)
        lon[a:b], lat[a:b] = D.tangent_grid_lonlat(cl[a:b], ca[a:b], (H, W), px, shift=shift)
        vals[a:b], valid[a:b] = D.sample_healpix(rgb, ids, LEVEL, lon[a:b], lat[a:b], interpolation="bilinear")
    cov = valid.reshape(M, -1).mean(1)
    if fill == "mean":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            m = np.nan_to_num(np.nanmean(vals.reshape(M, -1, 3), axis=1))
        vals = np.where(np.isfinite(vals), vals, m[:, None, None, :])
    return np.transpose(vals, (0, 3, 1, 2)).astype(np.float32), valid, cov, lon, lat

tiles, valid, cov, tlon, tlat = tangent_tiles_for(kids)
print(f"{len(kids)} images de {H} x {W} px = {H // 16} x {W // 16} tokens, pour des cellules de {2**k} px")

# [1] contrôle : mêmes pixels que D.tangent_tiles (une image, données = son parent et son anneau)
c0 = kids[len(kids) // 2]
n0 = D.cell_neighbours(PARENT_LEVEL, np.array([c0]))[:, 0]
m0 = np.isin(ids >> (2 * k), np.r_[c0, n0[n0 >= 0]])
rt, rc, *_ = D.tangent_tiles(rgb[m0], ids[m0], LEVEL, PARENT_LEVEL, tile_px=(H, W), gsd_m=gsd, fill="nan")
mine, _, _, lon0, lat0 = tangent_tiles_for(np.array([c0]), fill="nan")
inside0 = (D.cells_at_lonlat(LEVEL, lon0[0], lat0[0]) >> (2 * k)) == c0
ok = np.isfinite(rt[rc == c0][0]) & np.isfinite(mine[0]) & inside0[None]
print("[1] identique à D.tangent_tiles :", np.allclose(rt[rc == c0][0][ok], mine[0][ok], atol=1e-6))

def outline_px(cell, level, c_lon, c_lat, Hh, Ww, pxx, step=8):
    # contour d'une cellule HEALPix en coordonnées pixel (col, row) d'une image tangente
    blon, blat = D._boundaries_lonlat(level, np.array([cell]), step=step)
    xi, eta = D.lonlat_to_offsets(np.atleast_1d(c_lon), np.atleast_1d(c_lat), blon, blat)
    return np.c_[xi[0] / pxx + (Ww - 1) / 2, (Hh - 1) / 2 - eta[0] / pxx]

# vue nord-en-haut de la zone à la résolution native : référence des tests 1 et 3
OH, OW = D.parent_cover_px(np.array([zone]), ZONE_LEVEL, gsd, margin_px=16)
olon, olat = D.tangent_grid_lonlat(zlon, zlat, (OH, OW), px)
olon, olat = olon[0], olat[0]
ov, ov_ok = D.sample_healpix(rgb, ids, LEVEL, olon, olat, interpolation="bilinear")
zone_mask = D.cells_at_lonlat(ZONE_LEVEL, olon, olat) == zone       # pixels de la vue dans la zone
print(f"vue nord-en-haut de la zone : {OH} x {OW} px")

## Test 1 — la zone, et en dessous les images vues par DINO

* **(a)** bloc NESTED exact de la zone (`ZONE_PX` px). C'est la disposition de la grille du dessous : la case `(r, c)` est la cellule de l'image `(r, c)`. Le nord est vers le **coin haut-droit**.
* **(b)** même zone, nord en haut, avec les cellules numérotées `r,c`.
* **Grille** : l'image `(r, c)` telle que DINO la reçoit (nord en haut, après `fill="mean"`). Contour rouge : sa cellule. Il doit contenir le même paysage que la case `(r, c)` de (a), et occuper la même place que dans (b).

In [ ]:
blk, _, _, _ = D.nested_to_tiles(rgb[in_zone], ids[in_zone], LEVEL, ZONE_LEVEL, fill="nan")
S = blk.shape[-1]

fig = plt.figure(figsize=(16, 25))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.9], hspace=0.06, wspace=0.05)
ax = fig.add_subplot(gs[0, 0]); ax.imshow(show(blk[0]))
ax.set_title(f"(a) bloc NESTED {S}x{S} px de la zone (nord ↗)")
s8 = S / GRID
for i in range(1, GRID):
    ax.axhline(i * s8 - .5, c="w", lw=.4, alpha=.6); ax.axvline(i * s8 - .5, c="w", lw=.4, alpha=.6)
for r, c in zip(ROW, COL):
    ax.text((c + .5) * s8, (r + .5) * s8, f"{r},{c}", c="w", fontsize=max(4, 56 // GRID), ha="center", va="center")
ax.set_xticks([]); ax.set_yticks([])

ax = fig.add_subplot(gs[0, 1]); ax.imshow(show(ov, chw=False))
ax.contour(zone_mask, levels=[.5], colors="w", linewidths=1)
ax.set_title("(b) même zone, nord en haut")
for cell, r, c, cl_, ca_ in zip(kids, ROW, COL, clon, clat):
    p = outline_px(cell, PARENT_LEVEL, zlon, zlat, OH, OW, px)
    ax.add_patch(Polygon(p, fill=False, ec="r", lw=.5, alpha=.7))
    ax.text(*p.mean(0), f"{r},{c}", c="w", fontsize=max(4, 56 // GRID), ha="center", va="center")
ax.set_xlim(-.5, OW - .5); ax.set_ylim(OH - .5, -.5); ax.set_xticks([]); ax.set_yticks([])

sub = gs[1, :].subgridspec(GRID, GRID, hspace=0.15, wspace=0.04)
for t, (r, c) in enumerate(zip(ROW, COL)):
    ax = fig.add_subplot(sub[r, c])
    ax.imshow(show(tiles[t]), interpolation="nearest")
    ax.add_patch(Polygon(outline_px(kids[t], PARENT_LEVEL, clon[t], clat[t], H, W, px), fill=False, ec="r", lw=.6))
    ax.set_xlim(-.5, W - .5); ax.set_ylim(H - .5, -.5)
    ax.set_title(f"{r},{c}", fontsize=max(4, 48 // GRID), pad=1); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"zone {zone} — {GRID}x{GRID} images DINO de {H}x{W} px (cellules de {2**k} px, niveau {PARENT_LEVEL})", y=.9)
plt.show()

## Test 1b — contrôle chiffré

* **écart** : chaque pixel d'image comparé à la cellule HEALPix qui le contient. Faible et uniforme si tout va bien ; la même mesure sur l'image retournée haut/bas sert de référence.
* **part de la cellule dans l'image** : avec `TILE_PX = 2**k` (image de la même aire que la cellule), une cellule HEALPix n'est pas un carré et en déborde forcément. Ses tokens fins tombent alors hors de l'image et prennent le token du bord le plus proche.

In [ ]:
err, err_flip, frac_in = (np.full(len(kids), np.nan) for _ in range(3))
for t in range(len(kids)):
    near, nval = D.sample_healpix(rgb, ids, LEVEL, tlon[t], tlat[t], interpolation="nearest")
    tv, okp = np.transpose(tiles[t], (1, 2, 0)), valid[t] & nval
    if okp.any():
        err[t] = np.mean(np.abs(tv[okp] - near[okp]))
        err_flip[t] = np.mean(np.abs(tv[::-1][okp] - near[okp]))
    # part des pixels de la cellule (niveau LEVEL) qui tombent dans l'image
    ch = (kids[t] << (2 * k)) + np.arange(4 ** k)
    xi, eta = D.lonlat_to_offsets(clon[t:t+1], clat[t:t+1], *[v[None] for v in D.cell_centres_lonlat(LEVEL, ch)])
    cc, rr = xi[0] / px + (W - 1) / 2, (H - 1) / 2 - eta[0] / px
    frac_in[t] = np.mean((cc >= -.5) & (cc <= W - .5) & (rr >= -.5) & (rr <= H - .5))

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
ge = np.full((GRID, GRID), np.nan); ge[ROW, COL] = err
im = axs[0].imshow(ge); plt.colorbar(im, ax=axs[0]); axs[0].set_title("écart moyen image / cellule")
gf = np.full((GRID, GRID), np.nan); gf[ROW, COL] = frac_in
im = axs[1].imshow(gf, vmin=0.5, vmax=1, cmap="RdYlGn"); plt.colorbar(im, ax=axs[1])
axs[1].set_title("part de la cellule dans son image")
plt.show()
print(f"écart moyen {np.nanmean(err):.4f} (image retournée : {np.nanmean(err_flip):.4f}) "
      f"| dynamique {np.nanmedian(hi - lo):.4f}")
print(f"part de la cellule dans son image : min {frac_in.min():.1%}, moyenne {frac_in.mean():.1%}")

## Test 1c — zoom sur une image

Grille jaune : les patchs 16 × 16 px, un token chacun. Rouge : la cellule. Pointillés magenta : pixels remplis par `fill="mean"`.

In [ ]:
zr, zc = ZOOM_TILE if ZOOM_TILE is not None else (GRID // 2, GRID // 2)
t = int(np.flatnonzero((ROW == zr) & (COL == zc))[0])
fig, axs = plt.subplots(1, 2, figsize=(13, 6.5))
ax = axs[0]; ax.imshow(show(tiles[t]), interpolation="nearest")
for i in range(0, max(H, W) + 1, D.DINOV3_PATCH):
    ax.axhline(i - .5, c="y", lw=.6); ax.axvline(i - .5, c="y", lw=.6)
ax.add_patch(Polygon(outline_px(kids[t], PARENT_LEVEL, clon[t], clat[t], H, W, px), fill=False, ec="r", lw=1.5))
if (~valid[t]).any():
    ax.contour(~valid[t], levels=[.5], colors="m", linestyles=":", linewidths=.8)
ax.set_xlim(-.5, W - .5); ax.set_ylim(H - .5, -.5)
ax.set_title(f"image ({zr},{zc}) — ce que DINO reçoit ({H}x{W} px, nord en haut)")
mc = (ids >> (2 * k)) == kids[t]
b, _, _, _ = D.nested_to_tiles(rgb[mc], ids[mc], LEVEL, PARENT_LEVEL, fill="nan")
axs[1].imshow(show(b[0]), interpolation="nearest"); axs[1].set_title(f"la cellule en bloc NESTED {2**k} px (nord ↗)")
axs[1].set_xticks([]); axs[1].set_yticks([])
plt.show()

## Test 2 — sens du décalage des fenêtres `over_sample`

`GetDINOV3SAT(over_sample=n)` passe `n²` fenêtres décalées de `(a·16/n, b·16/n)` px, puis range leurs tokens aux indices fins
`[a::n, b::n]`. Ce rangement suppose qu'une fenêtre décalée de `dy > 0` part vers le **sud**, et de `dx > 0` vers l'**est**.

In [ ]:
step = D.DINOV3_PATCH // 2
base = tangent_tiles_for(kids, fill="nan")[0]
def mismatch(a, b, dr, dc):
    # écart moyen entre b[r, c] et a[r + dr, c + dc]
    A = a[..., max(dr, 0):H + min(dr, 0), max(dc, 0):W + min(dc, 0)]
    B = b[..., max(-dr, 0):H + min(-dr, 0), max(-dc, 0):W + min(-dc, 0)]
    return float(np.nanmean(np.abs(A - B)))

verdict = {}
for sh in [(step, 0), (0, step)]:
    v = tangent_tiles_for(kids, shift=sh, fill="nan")[0]
    att, opp = mismatch(base, v, *sh), mismatch(base, v, -sh[0], -sh[1])
    verdict[sh] = att < opp
    print(f"shift={sh}:  écart si sud/est (attendu) {att:.4f} | si nord/ouest {opp:.4f}  -> "
          f"{'OK' if att < opp else 'INVERSÉ'}")
ROW_SHIFT_OK, COL_SHIFT_OK = verdict[(step, 0)], verdict[(0, step)]

## Test 3 — `over_sample` de bout en bout

`GetDINOV3SAT(projection="tangent", over_sample=n, return_patches=True)` est appelé sur la zone (et un anneau de cellules autour),
avec les réglages des tests précédents (`tile_px=TILE_PX`). Les tokens sont rendus sur la vue nord-en-haut de la zone,
chacun à l'emplacement de sa cellule (niveau `LEVEL - 4 + log2 n`).

**3a — backbone exact.** Chaque token vaut la couleur moyenne de son patch 16 × 16. La réponse attendue est donc connue :
c'est l'image moyennée par une fenêtre glissante de 16 px, colonne « vérité ». Quand `n` augmente, la carte doit se
rapprocher de cette vérité, et l'écart chiffré diminuer. Si l'entrelacement est faux, on voit des lignes dédoublées ou en
zig-zag, et l'écart ne baisse plus.

**3b — DINOv3** (si `WEIGHTS`) : les mêmes cartes en ACP → RGB. La base de l'ACP est commune à toutes les vignettes.

In [ ]:
import torch, torch.nn as nn

class MeanDino(nn.Module):
    # « backbone » exact : token = moyenne RGB du patch 16x16, CLS = moyenne des tokens
    def __init__(self):
        super().__init__(); self.dummy = nn.Parameter(torch.zeros(1))
    def forward_features(self, x):
        tok = nn.functional.avg_pool2d(x, D.DINOV3_PATCH).flatten(2).transpose(1, 2)
        return {"x_norm_clstoken": tok.mean(1), "x_norm_patchtokens": tok}

_grid_pkg = D.tangent_grid_lonlat
def _grid_fixed(centre_lon, centre_lat, size, gsd_rad, *, shift=(0.0, 0.0), radius=D.EARTH_RADIUS_M):
    return _grid_pkg(centre_lon, centre_lat, size, gsd_rad, shift=(-shift[0], shift[1]), radius=radius)

@contextmanager
def shift_fix(on):
    # remplace provisoirement tangent_grid_lonlat par la version au décalage de ligne corrigé
    if on:
        D.tangent_grid_lonlat = _grid_fixed
    try:
        yield
    finally:
        D.tangent_grid_lonlat = _grid_pkg

VARIANTS = [("code du package", False)] + ([] if ROW_SHIFT_OK else [("décalage corrigé", True)])

# entrée : les cellules dont le parent est dans la zone ou dans un anneau assez large pour les images
ring = set(kids.tolist())
for _ in range(max(1, int(np.ceil(H / 2 ** k)))):
    nbr = D.cell_neighbours(PARENT_LEVEL, np.array(sorted(ring)))
    ring |= set(nbr[nbr >= 0].ravel().tolist())
m_in = np.isin(ids >> (2 * k), np.array(sorted(ring)))
d_in = np.where(np.isfinite(rgb[m_in]), rgb[m_in], np.nanmean(rgb[in_zone], 0))

def run(model, n, fixed, **kw):
    with shift_fix(fixed), warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        r = D.GetDINOV3SAT(d_in, ids[m_in], LEVEL, PARENT_LEVEL, projection="tangent", tile_px=TILE_PX,
                           over_sample=n, model=model, return_patches=True, min_coverage=0.0, **kw)
    out = [str(x.message) for x in w if "outside" in str(x.message)]
    return r, (out[0].split(" fall")[0] if out else "0")

def on_view(values, cell_id, level):
    # champ porté par des cellules -> vue nord-en-haut de la zone (plus proche voisin)
    v, _ = D.sample_healpix(values.astype(np.float32), cell_id, level, olon, olat, interpolation="nearest")
    return np.where(zone_mask[..., None], v, np.nan)

def box16(img):
    # moyenne glissante 16 x 16, ce que voit exactement un token centré sur le pixel
    s = D.DINOV3_PATCH
    m = np.isfinite(img).all(-1); x = np.where(m[..., None], img, 0.0)
    def cs(a):
        c = np.cumsum(np.cumsum(np.pad(a, ((1, 0), (1, 0)) + ((0, 0),) * (a.ndim - 2)), 0), 1)
        return c[s:, s:] - c[:-s, s:] - c[s:, :-s] + c[:-s, :-s]
    num, den = cs(x), cs(m.astype(float))
    out = np.full(img.shape, np.nan)
    with np.errstate(invalid="ignore", divide="ignore", over="ignore"):
        out[s // 2:s // 2 + num.shape[0], s // 2:s // 2 + num.shape[1]] = num / den[..., None]
    return out

truth = np.where(zone_mask[..., None], box16(ov), np.nan)
crop = np.ix_(np.any(zone_mask, 1), np.any(zone_mask, 0))
def view(a): return show(a[crop], chw=False)

res_mean = {}
for name, fixed in VARIANTS:
    for n in OVER_SAMPLES:
        r, nout = run(MeanDino(), n, fixed, device="cpu", mean=(0, 0, 0), std=(1, 1, 1))
        f = on_view(r.patch_embedding, r.patch_cell_id, r.patch_level)
        okm = np.isfinite(f).all(-1) & np.isfinite(truth).all(-1)
        res_mean[name, n] = f
        print(f"[{name:16s}] n={n}  tokens niveau {r.patch_level}  |token - vérité| = "
              f"{np.mean(np.abs(f[okm] - truth[okm])):.4f}   cellules hors image : {nout}")

cy, cx = np.argwhere(zone_mask).mean(0).astype(int)
ZOOM = (slice(cy - 96, cy + 96), slice(cx - 96, cx + 96))            # 192 x 192 px au centre de la zone
ncol = 2 + len(OVER_SAMPLES)
for label, cr, sz in [("zone entière", crop, 4.2), ("zoom 192 px au centre", ZOOM, 4.6)]:
    fig, axs = plt.subplots(len(VARIANTS), ncol, figsize=(sz * ncol, sz * 1.05 * len(VARIANTS)), squeeze=False)
    for i, (name, _) in enumerate(VARIANTS):
        axs[i, 0].imshow(show(np.where(zone_mask[..., None], ov, np.nan)[cr], chw=False), interpolation="nearest")
        axs[i, 0].set_title("image")
        axs[i, 1].imshow(show(truth[cr], chw=False), interpolation="nearest")
        axs[i, 1].set_title("vérité : moyenne glissante 16 px")
        for j, n in enumerate(OVER_SAMPLES):
            axs[i, 2 + j].imshow(show(res_mean[name, n][cr], chw=False), interpolation="nearest")
            axs[i, 2 + j].set_title(f"{name}\nover_sample={n}")
        for a in axs[i]: a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"3a — backbone exact (token = couleur moyenne du patch) — {label}", y=1.0)
    plt.tight_layout(); plt.show()

In [ ]:
# 3b — DINOv3, si les poids sont disponibles
if WEIGHTS is None:
    print("WEIGHTS = None : 3b sauté. Renseigner le chemin des poids DINOv3 SAT pour voir les vrais tokens.")
else:
    model = D.load_dinov3_sat(MODEL_NAME, weights=WEIGHTS)
    res_dino = {}
    for name, fixed in VARIANTS:
        for n in OVER_SAMPLES:
            r, nout = run(model, n, fixed, batch_size=64)
            res_dino[name, n] = r
            print(f"[{name:16s}] n={n}  {r.patch_embedding.shape[0]} tokens niveau {r.patch_level}")
    # base ACP commune : tokens de la zone, code du package, n = 1
    r1 = res_dino[VARIANTS[0][0], OVER_SAMPLES[0]]
    z1 = (r1.patch_cell_id >> (2 * (r1.patch_level - ZONE_LEVEL))) == zone
    mu = r1.patch_embedding[z1].mean(0)
    _, _, Vt = np.linalg.svd(r1.patch_embedding[z1] - mu, full_matrices=False)
    P = Vt[:3].T
    fields = {key: on_view((r.patch_embedding - mu) @ P, r.patch_cell_id, r.patch_level)
              for key, r in res_dino.items()}
    allv = np.concatenate([f[np.isfinite(f).all(-1)] for f in fields.values()])
    plo, phi = np.percentile(allv, [2, 98], axis=0)
    ncol = 1 + len(OVER_SAMPLES)
    for label, cr, sz in [("zone entière", crop, 4.2), ("zoom 192 px au centre", ZOOM, 4.6)]:
        fig, axs = plt.subplots(len(VARIANTS), ncol, figsize=(sz * ncol, sz * 1.05 * len(VARIANTS)), squeeze=False)
        for i, (name, _) in enumerate(VARIANTS):
            axs[i, 0].imshow(show(np.where(zone_mask[..., None], ov, np.nan)[cr], chw=False), interpolation="nearest")
            axs[i, 0].set_title("image")
            for j, n in enumerate(OVER_SAMPLES):
                f = fields[name, n][cr]
                axs[i, 1 + j].imshow(np.clip((np.nan_to_num(f, nan=0) - plo) / (phi - plo), 0, 1), interpolation="nearest")
                axs[i, 1 + j].set_title(f"{name}\nover_sample={n}")
            for a in axs[i]: a.set_xticks([]); a.set_yticks([])
        fig.suptitle(f"3b — DINOv3, ACP des tokens -> RGB (base commune) — {label}", y=1.0)
        plt.tight_layout(); plt.show()